# V1DD Functional Stimulus Metrics

Regenerating the `allen_v1dd` stimulus-response metrics — drifting gratings, surround
suppression, natural images, natural movie, receptive fields — from the NWB-Zarr
functional asset.

The original pipeline read a private Isilon HDF5 tree through `OPhysClient`, which no
longer exists; its own README says the code "would not work with the format you have
access to". So this is the same conversion as
**`Functional Data Cell-Cell Correlations.ipynb`**: keep the analysis, replace the data
access.

### Where this notebook currently is

| Milestone | Status |
|---|---|
| **M1 — schema truth** | this notebook |
| **M2 — response engine** | this notebook |
| **M3 — natural movie** (the deterministic end-to-end check) | this notebook |
| **M4 — drifting gratings → surround suppression** | this notebook |
| M5 — natural images / images 12 | not yet |
| M6 — receptive fields | not yet |
| M7 — packaging into the seven published tables | not yet |

M1 and M2 exist to *earn confidence before computing anything*. The per-trial stimulus
table this whole port depends on had never been read off a real file — its schema was
reconstructed from the NWB writer script — so M1 describes what is actually there. M2
proves the response arithmetic against a synthetic trace, where the right answer is known.

Each milestone writes a small JSON to `{save_dir}/checks/`, which is committed, so the
results can be reviewed away from the capsule.

In [1]:
import os
import sys
import time
from os.path import join as pjoin

import numpy as np
import pandas as pd
from IPython.display import display

# Robustly locate utils regardless of the kernel's working directory.
for _candidate in [pjoin("..", "utils"), pjoin("code", "utils"), "utils"]:
    if os.path.isdir(_candidate):
        sys.path.append(os.path.abspath(_candidate))
        break
else:
    raise FileNotFoundError(f"could not locate 'utils'; cwd={os.getcwd()}")

import stimulus_metrics as sm
import trial_responses as tr
import v1dd_nwb as vn
from checkpoints import checkpoint
from paths import resolve_data_root, resolve_dataset_dir

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
print(f"numpy {np.__version__} | pandas {pd.__version__}")

numpy 2.5.1 | pandas 2.3.3


## Paths

Input location comes from `utils/paths.py`, so this notebook finds its data on CodeOcean,
off the workshop USB drive, or in a local checkout without editing anything. Outputs
follow the correlations notebook's `scratch`/`results` knob.

In [2]:
mat_version = 1196

data_root = resolve_data_root(f"v1dd_{mat_version}")
functional_dir = resolve_dataset_dir("409828_V1DD_Filtered", root=data_root)

# The published allen_v1dd metric tables, for validation. Optional: without them the
# metrics still compute, they just cannot be checked against the original.
published_dir = resolve_dataset_dir("data_frames", root=data_root, required=False)

# Where derived tables and verification artifacts go.
output_target = "scratch"   # "scratch" (default, ephemeral) | "results" (reproducible run)
save_dir = pjoin(f"/{output_target}", f"v1dd_{mat_version}_coreg_functional_metrics")
os.makedirs(save_dir, exist_ok=True)

# The two sessions with EM coregistration. The correlations notebook derives this list
# live from CAVE; hard-coding it here keeps M1 free of a network dependency, and the
# schema report prints each session's own (column, volume) so a mismatch is visible.
TARGET_SESSIONS = [(1, "3"), (1, "5")]

print(f"data_root      : {data_root}")
print(f"functional_dir : {functional_dir}")
print(f"published_dir  : {published_dir}")
print(f"save_dir       : {save_dir}")

data_root      : /data/
functional_dir : /data/409828_V1DD_Filtered
published_dir  : /data/data_frames
save_dir       : /scratch/v1dd_1196_coreg_functional_metrics


## Session discovery

Each session's `(column, volume)` is read from the ROI table *inside* the file, never
from the directory name. If the correlations notebook has already been run, its cached
`session_index.csv` is reused — indexing all 23 sessions from scratch takes about five
minutes.

In [3]:
from pathlib import Path

session_paths = sorted(Path(functional_dir).glob("*/*.nwb.zarr"))
if not session_paths:
    session_paths = sorted(Path(functional_dir).rglob("*.nwb.zarr"))
if not session_paths:
    raise FileNotFoundError(f"no *.nwb.zarr under {functional_dir}")
print(f"{len(session_paths)} session(s) in the asset")

# Reuse the correlations notebook's cached index if it is available.
_cached = resolve_dataset_dir(
    f"v1dd_{mat_version}_coreg_functional_correlation", root=data_root, required=False
)
session_index = None
if _cached and os.path.isfile(pjoin(_cached, "session_index.csv")):
    session_index = pd.read_csv(pjoin(_cached, "session_index.csv"))
    session_index["volume"] = session_index["volume"].astype(str)
    print(f"reusing cached session index from {_cached}")
else:
    print("no cached index found; peeking at each session (~5 min) ...")
    rows = []
    for p in session_paths:
        try:
            nwb, io = vn.open_session(p)
            first = vn.list_planes(nwb)[0]
            rois = nwb.processing[first]["dff"].rois.to_dataframe()
            rows.append({
                "path": str(p), "name": p.parent.name,
                "column": int(pd.to_numeric(rois["column"]).iloc[0]),
                "volume": str(int(pd.to_numeric(rois["volume"]).iloc[0])),
                "n_planes": len(vn.list_planes(nwb)),
                "session_id": str(nwb.session_id),
            })
            io.close()
        except Exception as exc:
            print(f"  could not index {p.parent.name}: {type(exc).__name__}: {exc}")
    session_index = pd.DataFrame(rows)

session_index["target"] = [
    (int(c), str(v)) in TARGET_SESSIONS
    for c, v in zip(session_index["column"], session_index["volume"])
]
targets = session_index.loc[session_index["target"]].reset_index(drop=True)
display(session_index)
print(f"\n{len(targets)} target session(s):")
for _, r in targets.iterrows():
    print(f"  column {r['column']}, volume {r['volume']}  ->  {r['name']}")
if len(targets) != len(TARGET_SESSIONS):
    print(f"\n!! expected {len(TARGET_SESSIONS)} target sessions, found {len(targets)}")

23 session(s) in the asset
reusing cached session index from /data/v1dd_1196_coreg_functional_correlation


,path,name,column,volume,n_planes,session_id,coregistered,target
0,/data/409828_V1DD_Filtered/409828_2018-11-06_1...,409828_2018-11-06_14-02-59_filtered_2026-04-09...,2,1,6,774328450,False,False
1,/data/409828_V1DD_Filtered/409828_2018-11-20_1...,409828_2018-11-20_10-42-45_filtered_2026-04-09...,3,1,6,783110306,False,False
2,/data/409828_V1DD_Filtered/409828_2018-11-21_0...,409828_2018-11-21_09-22-23_filtered_2026-04-16...,4,1,6,783878040,False,False
3,/data/409828_V1DD_Filtered/409828_2018-11-21_1...,409828_2018-11-21_10-56-07_filtered_2026-04-09...,4,2,6,784057573,False,False
4,/data/409828_V1DD_Filtered/409828_2018-11-26_1...,409828_2018-11-26_11-16-25_filtered_2026-04-09...,5,1,6,785378984,False,False
5,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_11-01-58_filtered_2026-04-09...,2,2,6,785941763,False,False
6,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_12-29-05_filtered_2026-04-09...,2,3,6,786071018,False,False
7,/data/409828_V1DD_Filtered/409828_2018-11-28_1...,409828_2018-11-28_10-54-56_filtered_2026-04-16...,3,3,6,786879416,False,False
8,/data/409828_V1DD_Filtered/409828_2018-11-29_1...,409828_2018-11-29_13-42-04_filtered_2026-04-09...,5,2,6,788220278,False,False
9,/data/409828_V1DD_Filtered/409828_2018-12-03_1...,409828_2018-12-03_14-25-24_filtered_2026-04-09...,4,3,6,790009715,False,False



2 target session(s):
  column 1, volume 3  ->  409828_2018-12-13_15-10-05_filtered_2026-04-09_05-57-20
  column 1, volume 5  ->  409828_2018-12-14_14-47-35_filtered_2026-04-09_06-13-08


## M1 — schema truth

`schema_report()` describes a session without computing any metric. It **reports rather
than asserts**: a missing table or column is recorded as an error string, because the
point is to learn what the file contains, and a function that raises on the first
surprise tells you much less than one that describes the whole file.

The things this needs to settle:

* Does `intervals['stimulus_table']` have the twelve expected columns? This schema was
  read off the NWB *writer* script and has never been verified against a real file.
  Everything downstream depends on it.
* Do the per-family trial counts match the denominators visible in the published CSVs —
  drifting gratings 8, natural images 8, natural images 12 **40**, natural movie **9**?
* Are there exactly 12 grating directions? The original hard-codes `(dir ± 3) % 12` for
  orthogonal directions and `(dir + 6) % 12` for the null direction, so any other number
  silently computes the wrong metric.
* Is the locally-sparse-noise template a 2× upsample of the 8×14 grid the receptive-field
  code was written against? **This decides whether M6 is possible at all.**
* Is `stop_time - start_time` equal to the 2.0 s the original took from an NWB attribute?
  If not, the response window is a decision rather than a lookup.

In [4]:
%%time
reports = {}
for _, row in targets.iterrows():
    key = f"col{row['column']}_vol{row['volume']}"
    print(f"--- {key}  ({row['name']})")
    nwb, io = vn.open_session(row["path"])
    try:
        reports[key] = vn.schema_report(nwb)
        reports[key]["session"] = {
            "name": row["name"], "session_id": str(row["session_id"]),
            "column": int(row["column"]), "volume": str(row["volume"]),
        }
    finally:
        io.close()

path = checkpoint(
    "schema_report", reports, save_dir,
    sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()],
)

--- col1_vol3  (409828_2018-12-13_15-10-05_filtered_2026-04-09_05-57-20)
CPU times: user 6.31 s, sys: 361 ms, total: 6.67 s
Wall time: 48.5 s


KeyboardInterrupt: 

In [ ]:
# Compact verdict table -- the questions M1 exists to answer.
EXPECTED_TRIALS = {"drifting_gratings_full": 8, "drifting_gratings_windowed": 8,
                   "natural_images": 8, "natural_images_12": 40, "natural_movie": 9}

rows = []
for key, rep in reports.items():
    st = rep.get("stimulus_table", {})
    ps = rep.get("per_stimulus", {})
    lsn = rep.get("lsn_template", {})
    dgf = ps.get("drifting_gratings_full", {})

    rows.append({"session": key, "question": "stimulus_table present",
                 "answer": "error" not in st, "detail": st.get("error", f"{st.get('n_rows')} rows")})
    rows.append({"session": key, "question": "all 12 expected columns",
                 "answer": st.get("missing_vs_expected") == [],
                 "detail": f"missing={st.get('missing_vs_expected')} extra={st.get('extra_vs_expected')}"})
    rows.append({"session": key, "question": "exactly 12 grating directions",
                 "answer": dgf.get("n_directions") == 12,
                 "detail": str(dgf.get("n_directions"))})
    for fam, want in EXPECTED_TRIALS.items():
        got = (ps.get(fam) or {}).get("n_trials_inferred")
        rows.append({"session": key, "question": f"n_trials[{fam}] == {want}",
                     "answer": got == want, "detail": str(got)})
    rows.append({"session": key, "question": "one spontaneous block",
                 "answer": rep.get("epochs", {}).get("n_spontaneous_blocks") == 1,
                 "detail": str(rep.get("epochs", {}).get("n_spontaneous_blocks"))})
    rows.append({"session": key, "question": "is_soma == pika_conf > 0.5",
                 "answer": all(d.get("is_soma_matches_conf_gt_0.5") is True
                               for d in rep.get("planes", {}).get("detail", {}).values()),
                 "detail": ""})
    rows.append({"session": key, "question": "LSN template reduces to 8x14 (M6 viable)",
                 "answer": lsn.get("rf_viable") is True,
                 "detail": f"{lsn.get('native_shape')} -> {lsn.get('final_shape')}; "
                           f"uniform={lsn.get('blocks_uniform')}; {lsn.get('error', '')}"})
    dur = dgf.get("sweep_duration_s", {})
    rows.append({"session": key, "question": "DG stop-start == 2.0 s",
                 "answer": abs((dur.get("median_stop_minus_start") or 0) - 2.0) < 0.01,
                 "detail": f"stop-start={dur.get('median_stop_minus_start')}, "
                           f"onset-to-onset={dur.get('median_onset_to_onset')}"})

verdict = pd.DataFrame(rows)
display(verdict)
n_bad = int((~verdict["answer"].astype(bool)).sum())
print(f"\n{len(verdict) - n_bad}/{len(verdict)} checks answered as expected")
if n_bad:
    print("\nUnexpected answers -- read these before continuing to M3:")
    display(verdict.loc[~verdict["answer"].astype(bool)])

In [ ]:
# The stimulus table itself, for eyeballing. This is the artifact of record for M1.
nwb, io = vn.open_session(targets.iloc[0]["path"])
try:
    stim_table = vn.load_stimulus_table(nwb)
    epochs = vn.epoch_table(nwb)
finally:
    io.close()

print(f"stimulus_table: {stim_table.shape}")
display(stim_table.head(8))
print("\nsweeps per stimulus:")
display(stim_table["stim_name"].value_counts().rename("n_sweeps").to_frame())
print("\nepochs:")
display(epochs)

## M2 — response engine

`trial_responses.py` is the arithmetic layer: given traces, timestamps and stimulus onsets,
what was each neuron's mean activity in a window? The original asked this with a Python
loop over every sweep and every bootstrap draw, which costs 40–50 minutes for these two
sessions. Replacing it with a **prefix sum over time** makes each window mean two array
lookups, and the port runs in about five.

The subtlety worth stating: response windows land on a *variable* number of imaging
frames, because stimulus onsets are not frame-aligned. That looks like it forces a loop.
It does not — the samples are never materialised, so `b - a` is just an integer vector.

These checks use a synthetic trace where the right answer is known, so they prove the
engine independently of the data. Two of them are load-bearing:

* **Label-closed windows.** The original selected with `xarray.sel(time=slice(...))`,
  which includes *both* endpoints. A natural `(t >= lo) & (t < hi)` drops one sample per
  trial and shifts every response — the kind of difference that survives into a metric and
  looks like an algorithm bug.
* **Two different window primitives.** Trials use the label-closed form above; the
  bootstrap null uses a *frame-indexed*, fixed-width slice of `round(w / dt)` samples. At
  dt ≈ 0.164 s a 2 s grating window gives 13 samples for a trial and 12 for a null draw.
  That asymmetry is in the original, and reproducing its numbers means reproducing it.

In [ ]:
checks = {}
_rng = np.random.default_rng(0)
_n, _dt = 2000, 0.16374
_ts = np.cumsum(_rng.normal(_dt, _dt * 0.002, _n)) + 12.3      # jittered, like a real clock
_traces = _rng.gamma(2.0, 0.5, size=(_n, 7))                   # events-like, non-negative
_starts = _rng.uniform(_ts[5], _ts[-30], size=200)

# 1. label-closed windows, against the definition
_bad = 0
for w0, w1 in [(0.0, 2.0), (-1.0, 0.0), (0.0, 3 * _dt)]:
    a, b = tr.window_bounds(_ts, _starts, w0, w1)
    for i, s in enumerate(_starts):
        want = np.flatnonzero((_ts >= s + w0) & (_ts <= s + w1))   # BOTH ends inclusive
        if not np.array_equal(want, np.arange(a[i], b[i])):
            _bad += 1
checks["window_bounds_label_closed"] = {"mismatches": int(_bad), "n_tested": 600}

a, b = tr.window_bounds(_ts, _starts, 0.0, 2.0)
checks["window_width_varies"] = {"widths": sorted(int(w) for w in np.unique(b - a))}

# 2. prefix-sum means == direct slicing
cs, counts = tr.prefix_sums(_traces)
got = tr.window_means(cs, counts, a, b)
want = np.stack([_traces[a[i]:b[i]].mean(axis=0) for i in range(len(_starts))])
checks["window_means_vs_direct"] = {"max_abs_diff": float(np.max(np.abs(got - want)))}

# 3. nan-aware means
_tn = _traces.copy()
_tn[_rng.random(_tn.shape) < 0.02] = np.nan
csn, cn = tr.prefix_sums(_tn)
gotn = tr.window_means(csn, cn, a, b)
wantn = np.stack([np.nanmean(_tn[a[i]:b[i]], axis=0) for i in range(len(_starts))])
checks["nan_aware_means"] = {"max_abs_diff": float(np.nanmax(np.abs(gotn - wantn))),
                             "counts_array_built": cn is not None}

# 4. bootstrap null: fixed-width frame windows, and reproducible
_null = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                            n_boot=500, rng=np.random.default_rng(42))
_again = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                             n_boot=500, rng=np.random.default_rng(42))
checks["spontaneous_null"] = {
    "shape": list(_null.shape),
    "reproducible_under_seed": bool(np.array_equal(_null, _again)),
    "null_window_samples": int(round(2.0 / float(np.median(np.diff(_ts))))),
    "trial_window_samples_median": int(np.median(b - a)),
}

# 5. trial scatter, NaN padding, chronological rank
_resp = np.arange(16, dtype=float).reshape(8, 2)
_cond = np.array([0, 1, 2, 0, 1, 2, 0, 2])
_ta = tr.trial_array(_resp, _cond, n_trials=3, n_conditions=3)
checks["trial_array"] = {
    "shape": list(_ta.shape),
    "chronological_within_condition": bool(np.array_equal(_ta[0, :, 0], _resp[[0, 3, 6], 0])),
    "short_condition_nan_padded": bool(np.isnan(_ta[1, 2, 0])),
}

# 6. lifetime sparseness anchors
checks["lifetime_sparseness"] = {
    "uniform_is_0": float(tr.lifetime_sparseness(np.ones((1, 20)))[0]),
    "one_hot_is_1": float(tr.lifetime_sparseness(np.eye(1, 20)) [0]),
}

# 7. frac_trials_above_null excludes NaN trials rather than scoring them
_nl = np.tile(np.linspace(0, 1, 1000), (2, 1))
_trials = np.array([[2.0, 2.0, 2.0, 2.0], [0.5, 2.0, np.nan, np.nan]])
_fr = tr.frac_trials_above_null(_trials, _nl)
checks["frac_trials_above_null"] = {"all_strong": float(_fr[0]), "mixed_with_nan": float(_fr[1])}

ok = (checks["window_bounds_label_closed"]["mismatches"] == 0
      and checks["window_means_vs_direct"]["max_abs_diff"] < 1e-9
      and checks["nan_aware_means"]["max_abs_diff"] < 1e-9
      and checks["spontaneous_null"]["reproducible_under_seed"]
      and checks["trial_array"]["chronological_within_condition"]
      and checks["trial_array"]["short_condition_nan_padded"]
      and abs(checks["lifetime_sparseness"]["uniform_is_0"]) < 1e-12
      and abs(checks["lifetime_sparseness"]["one_hot_is_1"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["all_strong"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["mixed_with_nan"] - 0.5) < 1e-12)
checks["all_passed"] = bool(ok)

for k, v in checks.items():
    print(f"  {k}: {v}")
print(f"\nengine checks: {'ALL PASSED' if ok else 'FAILED -- do not continue to M3'}")

checkpoint("engine_tests", checks, save_dir, seed=0)

## M3 — natural movie

The first metric family, and deliberately the one with no bootstrap in it.

`frac_responsive_trials` for natural movie is not a statistical test: it is the fraction
of movie repeats whose mean response at the neuron's preferred frame is strictly greater
than zero. So comparing it against the published table exercises the stimulus table, the
trial scatter, the NaN padding, the response window and the argmax **with zero
stochasticity** — a hard pass/fail on the entire extraction path before any randomness is
introduced. `pref_img`, `pref_response` and `lifetime_sparseness` are equally
deterministic. Only `z_score` involves the bootstrap.

Two things about this family that are the original's design rather than ours: each movie
frame counts as a trial, and the response window spans about three imaging frames
(~0.49 s) while movie frames are 1/30 s apart — so consecutive "trials" overlap heavily.
`lifetime_sparseness` over 3,600 x 9 such values is therefore not measuring what its name
suggests. We reproduce it because the goal is to match the published table.

In [4]:
%%time
rng_seed = 0
nm_tables = []

for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        trials, _ = vn.stimulus_trials(stim, "natural_movie")
        spont = vn.spontaneous_block(nwb)
        for plane_key in vn.list_planes(nwb):
            plane = vn.load_plane(nwb, plane_key, trace_types=("events",))
            nm_tables.append(sm.natural_movie_metrics(
                plane, trials, spont,
                rng=np.random.default_rng(rng_seed),
            ))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: "
                  f"{plane.n_rois} ROIs, {len(trials)} sweeps")
            del plane
    finally:
        io.close()

nm_new = pd.concat(nm_tables, ignore_index=True)
print()
print(f"natural_movie: {len(nm_new)} ROIs across {len(nm_tables)} planes")
display(nm_new.head())

  col1 vol3 plane-0: 409 ROIs, 29700 sweeps
  col1 vol3 plane-1: 470 ROIs, 29700 sweeps
  col1 vol3 plane-2: 483 ROIs, 29700 sweeps
  col1 vol3 plane-3: 478 ROIs, 29700 sweeps
  col1 vol3 plane-4: 438 ROIs, 29700 sweeps
  col1 vol3 plane-5: 430 ROIs, 29700 sweeps
  col1 vol5 plane-0: 193 ROIs, 29700 sweeps
  col1 vol5 plane-1: 228 ROIs, 29700 sweeps
  col1 vol5 plane-2: 202 ROIs, 29700 sweeps
  col1 vol5 plane-3: 131 ROIs, 29700 sweeps
  col1 vol5 plane-4: 90 ROIs, 29700 sweeps
  col1 vol5 plane-5: 121 ROIs, 29700 sweeps

natural_movie: 3673 ROIs across 12 planes


,roi_unique_id,roi_key,mouse,column,volume,plane,roi,frac_responsive_trials,lifetime_sparseness,pref_img,pref_response,z_score
0,M409828_3_0_0,M409828_13_0_0,M409828,1,3,0,0,1.000,0.954720,3359,0.104166,194.392479
1,M409828_3_0_1,M409828_13_0_1,M409828,1,3,0,1,1.000,0.970111,3355,0.115470,169.656202
2,M409828_3_0_2,M409828_13_0_2,M409828,1,3,0,2,0.625,0.929998,2703,0.016202,13.624954
3,M409828_3_0_3,M409828_13_0_3,M409828,1,3,0,3,1.000,0.967538,1911,0.110897,111.641997
4,M409828_3_0_4,M409828_13_0_4,M409828,1,3,0,4,0.875,0.944891,1632,0.063971,54.152452


CPU times: user 15.6 s, sys: 4.13 s, total: 19.8 s
Wall time: 1min 39s


In [5]:
# Compare against the published table. Joined on (column, volume, plane, roi) --
# never on roi_unique_id, which omits the column and collides across the five columns.
NM_METRICS = ["frac_responsive_trials", "lifetime_sparseness", "pref_img",
              "pref_response", "z_score"]

if published_dir is None:
    print("no published tables attached; skipping validation")
    nm_report = {"skipped": "published_dir not found"}
else:
    nm_pub = sm.load_published(published_dir, "natural_movie")
    nm_report = sm.compare_to_published(
        sm.to_published_schema(nm_new, "natural_movie"), nm_pub,
        NM_METRICS, exact=["pref_img"],
    )
    print(f"joined {nm_report['n_joined']} of {nm_report['n_new']} regenerated ROIs "
          f"({nm_report['n_only_published']} published rows have no NWB counterpart)")
    rows = []
    for m, v in nm_report["metrics"].items():
        rows.append({"metric": m, "n": v.get("n_both_finite"),
                     "max_abs_diff": v.get("max_abs_diff"),
                     "median_abs_diff": v.get("median_abs_diff"),
                     "frac_within_1e-9": v.get("frac_within_tol"),
                     "frac_exact": v.get("frac_exact"),
                     "pearson_r": v.get("pearson_r")})
    display(pd.DataFrame(rows))

    # The gate: frac_responsive_trials has no bootstrap, so it must match essentially
    # exactly. Anything else means the extraction path is wrong.
    gate = nm_report["metrics"]["frac_responsive_trials"]
    ok = gate.get("max_abs_diff", np.inf) < 1e-9
    print()
    print(f"GATE  frac_responsive_trials max_abs_diff = {gate.get('max_abs_diff')}"
          f"  ->  {'PASS' if ok else 'FAIL'}")
    if not ok:
        print("  The extraction path disagrees with the original. Suspects, in order:")
        print("   - response window (3 x imaging frame period)")
        print("   - trial scatter / NaN padding")
        print("   - argmax over frames picking a different preferred frame")

joined 3673 of 3673 regenerated ROIs (159633 published rows have no NWB counterpart)


,metric,n,max_abs_diff,median_abs_diff,frac_within_1e-9,frac_exact,pearson_r
0,frac_responsive_trials,3673,0.000000e+00,0.000000e+00,1.000000,NaN,1.000000
1,lifetime_sparseness,3673,1.619226e-09,1.718473e-10,0.988021,NaN,1.000000
2,pref_img,3673,0.000000e+00,0.000000e+00,1.000000,1.0,1.000000
3,pref_response,3673,6.674478e-09,9.701277e-11,0.968418,NaN,1.000000
4,z_score,3673,9.228563e+00,1.468847e-01,0.000000,NaN,0.999746



GATE  frac_responsive_trials max_abs_diff = 0.0  ->  PASS


In [6]:
nm_out = sm.to_published_schema(nm_new, "natural_movie")
nm_path = pjoin(save_dir, "natural_movie_M409828.csv")
nm_out.to_csv(nm_path, index=False)
print(f"wrote {nm_path}  ({len(nm_out)} rows)")

checkpoint("nm_validation", nm_report, save_dir, seed=rng_seed,
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

# Per-ROI differences, for drilling into any metric that disagrees.
if published_dir is not None:
    merged = nm_out.merge(
        sm.load_published(published_dir, "natural_movie"),
        on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
    keep = ["column", "volume", "plane", "roi"]
    for m in NM_METRICS:
        merged[f"{m}_diff"] = (pd.to_numeric(merged[f"{m}_new"], errors="coerce")
                               - pd.to_numeric(merged[f"{m}_pub"], errors="coerce"))
        keep += [f"{m}_new", f"{m}_pub", f"{m}_diff"]
    per_roi = pjoin(save_dir, "checks", "nm_per_roi.csv")
    merged[keep].to_csv(per_roi, index=False)
    print(f"wrote {per_roi}  ({len(merged)} rows)")

wrote /scratch/v1dd_1196_coreg_functional_metrics/natural_movie_M409828.csv  (3673 rows)
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/nm_validation.json  (1.9 KB)
wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/nm_per_roi.csv  (3673 rows)


## M4 — drifting gratings, then surround suppression

The biggest family, and the first one where the bootstrap reaches a *published* column.

Natural movie's `frac_responsive_trials` was `mean(response > 0)` — no randomness. Here it
is the fraction of preferred-condition trials beating a 2,500-draw spontaneous null at
p < 0.05, and `is_responsive` thresholds that at 0.5. So ROIs sitting near the boundary
will flip between random seeds, and there is no way to tell a flip from a bug without
knowing how much the metric moves on its own.

Hence the **two-seed control**: everything is computed twice with different seeds, and the
seed-A-vs-seed-B agreement is reported beside the seed-A-vs-published agreement. Only a
metric that agrees with the published table *materially worse than it agrees with itself*
is evidence of a problem.

Three things to watch:

* **`preferred_dir` / `preferred_sf`** are deterministic — they should match near-exactly.
  They also drive surround suppression, so a disagreement here propagates.
* **`osi` / `dsi` / `gosi` / `pref_dir_mean` / `lifetime_sparseness`** are deterministic
  given the trial responses. Expect float round-off, as in M3.
* **`frac_responsive_trials` / `is_responsive`** are the stochastic pair. Read them against
  the seed control, not against zero.

### The response-window decision

M1 measured drifting-grating sweeps at `stop_time - start_time` = **1.985 s**, while the
original took **2.0 s** from an NWB attribute it no longer has access to. `MetricConfig`
defaults to 2.0 to reproduce the published numbers. The cell after the comparison reruns
one plane at 1.985 s so the size of that choice is visible rather than assumed.

In [4]:
%%time
DG_METRICS = ["dsi", "frac_responsive_trials", "gosi", "is_responsive",
              "lifetime_sparseness", "osi", "preferred_dir", "preferred_sf",
              "pref_dir_mean"]
SEEDS = (0, 1)          # two seeds: the second is the noise-floor control

dg_tables = {s: {"full": [], "windowed": []} for s in SEEDS}
ssi_tables = {s: [] for s in SEEDS}

for _, srow in targets.iterrows():
    nwb, io = vn.open_session(srow["path"])
    try:
        stim = vn.load_stimulus_table(nwb)
        spont = vn.spontaneous_block(nwb)
        running = vn.load_running_speed(nwb)
        trials = {}
        for dg_type in ("full", "windowed"):
            trials[dg_type] = vn.stimulus_trials(
                stim, f"drifting_gratings_{dg_type}", vn.DG_PARAM_COLUMNS)

        for plane_key in vn.list_planes(nwb):
            plane = vn.load_plane(nwb, plane_key, trace_types=("events",))
            for seed in SEEDS:
                # Tuning-curve fits depend only on the trial means, so they are
                # deterministic; refitting them for the control seed would double the
                # dominant cost (~29k curve_fit calls) for no information. ssi_tuning_fit
                # is therefore computed on the first seed only.
                cfg = sm.MetricConfig(fit_tuning_curves=(seed == SEEDS[0]))
                res = {}
                for dg_type in ("full", "windowed"):
                    t, blank = trials[dg_type]
                    res[dg_type] = sm.drifting_gratings_metrics(
                        plane, t, blank, spont, running, dg_type=dg_type, config=cfg,
                        rng=np.random.default_rng(seed))
                    dg_tables[seed][dg_type].append(res[dg_type].metrics)
                ssi_tables[seed].append(sm.surround_suppression_metrics(
                    res["windowed"], res["full"], plane))
            print(f"  col{plane.column} vol{plane.volume} {plane_key}: {plane.n_rois} ROIs")
            del plane
    finally:
        io.close()

dg_new = {s: {k: pd.concat(v, ignore_index=True) for k, v in d.items()}
          for s, d in dg_tables.items()}
ssi_new = {s: pd.concat(v, ignore_index=True) for s, v in ssi_tables.items()}
print()
print(f"drifting gratings: {len(dg_new[SEEDS[0]]['full'])} ROIs per stimulus type")
display(dg_new[SEEDS[0]]["windowed"].head())

  col1 vol3 plane-0: 409 ROIs
  col1 vol3 plane-1: 470 ROIs
  col1 vol3 plane-2: 483 ROIs
  col1 vol3 plane-3: 478 ROIs
  col1 vol3 plane-4: 438 ROIs
  col1 vol3 plane-5: 430 ROIs
  col1 vol5 plane-0: 193 ROIs
  col1 vol5 plane-1: 228 ROIs
  col1 vol5 plane-2: 202 ROIs
  col1 vol5 plane-3: 131 ROIs
  col1 vol5 plane-4: 90 ROIs
  col1 vol5 plane-5: 121 ROIs

drifting gratings: 3673 ROIs per stimulus type


,roi_unique_id,roi_key,mouse,column,volume,plane,roi,dsi,frac_responsive_trials,gosi,is_responsive,lifetime_sparseness,osi,preferred_dir,preferred_sf,pref_dir_mean
0,M409828_3_0_0,M409828_13_0_0,M409828,1,3,0,0,0.239718,0.166667,0.049596,0.0,0.682981,0.187371,30.0,0.04,6.878321
1,M409828_3_0_1,M409828_13_0_1,M409828,1,3,0,1,0.307083,0.250000,0.074137,0.0,0.681635,0.490517,0.0,0.04,7.964159
2,M409828_3_0_2,M409828_13_0_2,M409828,1,3,0,2,0.605679,0.000000,0.254687,0.0,0.799431,0.415835,210.0,0.08,285.264594
3,M409828_3_0_3,M409828_13_0_3,M409828,1,3,0,3,0.350194,0.125000,0.243435,0.0,0.698661,0.620734,120.0,0.04,122.003812
4,M409828_3_0_4,M409828_13_0_4,M409828,1,3,0,4,0.198711,0.000000,0.111571,0.0,0.715844,0.227240,210.0,0.04,274.544878


CPU times: user 23min 20s, sys: 3.36 s, total: 23min 24s
Wall time: 24min 46s


In [5]:
# Agreement with the published table, beside agreement between the two seeds.
def agreement_table(new_a, new_b, published, metrics, exact=()):
    """One row per metric: how well it matches the original, and how well it matches
    itself under a different random seed. The second column is the noise floor."""
    vs_pub = sm.compare_to_published(new_a, published, metrics, exact=exact)
    vs_seed = sm.compare_to_published(new_a, new_b, metrics, exact=exact)
    rows = []
    for m in metrics:
        p, s = vs_pub["metrics"][m], vs_seed["metrics"][m]
        rows.append({
            "metric": m,
            "n": p.get("n_both_finite"),
            "vs_published_median": p.get("median_abs_diff"),
            "vs_seed_median": s.get("median_abs_diff"),
            "vs_published_max": p.get("max_abs_diff"),
            "vs_seed_max": s.get("max_abs_diff"),
            "r_published": p.get("pearson_r"),
            "exact_published": p.get("frac_exact"),
        })
    return pd.DataFrame(rows), vs_pub, vs_seed


dg_reports, ssi_report = {}, None
if published_dir is None:
    print("no published tables attached; skipping validation")
else:
    for dg_type in ("full", "windowed"):
        fam = f"drifting_gratings_{dg_type}"
        pub = sm.load_published(published_dir, fam)
        tbl, vs_pub, vs_seed = agreement_table(
            sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam),
            sm.to_published_schema(dg_new[SEEDS[1]][dg_type], fam),
            pub, DG_METRICS, exact=["preferred_dir", "preferred_sf", "is_responsive"])
        dg_reports[fam] = {"vs_published": vs_pub, "vs_seed": vs_seed}
        print(f"--- {fam}  (joined {vs_pub['n_joined']} ROIs)")
        display(tbl)

    pub_ssi = sm.load_published(published_dir, "surround_supression_index")
    tbl, vs_pub, vs_seed = agreement_table(
        sm.to_published_schema(ssi_new[SEEDS[0]], "surround_supression_index"),
        sm.to_published_schema(ssi_new[SEEDS[1]], "surround_supression_index"),
        pub_ssi, sm.SSI_COLUMNS)
    ssi_report = {"vs_published": vs_pub, "vs_seed": vs_seed}
    print(f"--- surround suppression  (joined {vs_pub['n_joined']} ROIs)")
    display(tbl)

--- drifting_gratings_full  (joined 3673 ROIs)


,metric,n,vs_published_median,vs_seed_median,vs_published_max,vs_seed_max,r_published,exact_published
0,dsi,3673,5.131345e-09,0.0,3.778313e-08,0.0,1.000000,NaN
1,frac_responsive_trials,3673,0.000000e+00,0.0,5.000000e-01,0.5,0.982559,NaN
2,gosi,3673,2.228761e-09,0.0,1.897423e-08,0.0,1.000000,NaN
3,is_responsive,3673,0.000000e+00,0.0,1.000000e+00,1.0,0.958016,0.983665
4,lifetime_sparseness,3673,7.780477e-10,0.0,1.278176e-08,0.0,1.000000,NaN
5,osi,3673,3.978402e-09,0.0,3.945698e-08,0.0,1.000000,NaN
6,preferred_dir,3673,0.000000e+00,0.0,0.000000e+00,0.0,1.000000,1.000000
7,preferred_sf,3673,2.775558e-17,0.0,5.551115e-17,0.0,1.000000,0.000000
8,pref_dir_mean,3673,6.969042e-07,0.0,8.062127e-05,0.0,1.000000,NaN


--- drifting_gratings_windowed  (joined 3673 ROIs)


,metric,n,vs_published_median,vs_seed_median,vs_published_max,vs_seed_max,r_published,exact_published
0,dsi,3673,5.076350e-09,0.0,4.422320e-08,0.0,1.000000,NaN
1,frac_responsive_trials,3673,0.000000e+00,0.0,5.000000e-01,0.5,0.976712,NaN
2,gosi,3673,2.351179e-09,0.0,3.075596e-08,0.0,1.000000,NaN
3,is_responsive,3673,0.000000e+00,0.0,1.000000e+00,1.0,0.944639,0.978492
4,lifetime_sparseness,3673,8.856536e-10,0.0,2.087126e-08,0.0,1.000000,NaN
5,osi,3673,4.104823e-09,0.0,4.353293e-08,0.0,1.000000,NaN
6,preferred_dir,3673,0.000000e+00,0.0,0.000000e+00,0.0,1.000000,1.000000
7,preferred_sf,3673,2.775558e-17,0.0,5.551115e-17,0.0,1.000000,0.000000
8,pref_dir_mean,3673,7.019476e-07,0.0,1.374414e-04,0.0,1.000000,NaN


--- surround suppression  (joined 3673 ROIs)


,metric,n,vs_published_median,vs_seed_median,vs_published_max,vs_seed_max,r_published,exact_published
0,ssi,3673,5.061863e-09,0.0,4.875922e-08,0.0,1.000000,None
1,ssi_avg,3673,1.743675e-09,0.0,2.772713e-08,0.0,1.000000,None
2,ssi_avg_at_pref_sf,3673,2.282460e-09,0.0,3.316924e-08,0.0,1.000000,None
3,ssi_running,730,4.280748e-09,0.0,4.310314e-08,0.0,1.000000,None
4,ssi_running_avg_at_pref_sf,3556,3.024742e-09,0.0,5.643858e-08,0.0,1.000000,None
5,ssi_stationary,2631,5.541394e-09,0.0,6.584289e-08,0.0,1.000000,None
6,ssi_stationary_avg_at_pref_sf,3673,2.809451e-09,0.0,2.984942e-08,0.0,1.000000,None
7,ssi_tuning_fit,3576,4.376523e-06,NaN,6.439872e-01,NaN,0.997447,None


In [6]:
# is_responsive is a threshold on a stochastic quantity, so report the confusion matrix
# rather than a correlation -- a 2% disagreement means different cells, not a smaller number.
if published_dir is not None:
    for dg_type in ("full", "windowed"):
        fam = f"drifting_gratings_{dg_type}"
        merged = sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam).merge(
            sm.load_published(published_dir, fam),
            on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
        a = merged["is_responsive_new"].fillna(0) > 0.5
        b = merged["is_responsive_pub"].fillna(0) > 0.5
        print(f"{fam}: agree {np.mean(a == b):.3%}  "
              f"(both yes {int((a & b).sum())}, both no {int((~a & ~b).sum())}, "
              f"new-only {int((a & ~b).sum())}, pub-only {int((~a & b).sum())})")
        # the same confusion, but between our two seeds -- the noise floor
        a2 = sm.to_published_schema(dg_new[SEEDS[1]][dg_type], fam)["is_responsive"] > 0.5
        print(f"{'':>{len(fam)}}  seed-to-seed agreement {np.mean(a.to_numpy() == a2.to_numpy()):.3%}")

drifting_gratings_full: agree 98.366%  (both yes 941, both no 2672, new-only 34, pub-only 26)
                        seed-to-seed agreement 98.394%
drifting_gratings_windowed: agree 97.849%  (both yes 929, both no 2665, new-only 45, pub-only 34)
                            seed-to-seed agreement 98.230%


In [7]:
# What the response-window choice is worth. M1 measured stop-start at 1.985 s; the
# original used 2.0 s from an NWB attribute. One plane, both windows, same seed.
_row = targets.iloc[0]
nwb, io = vn.open_session(_row["path"])
try:
    stim = vn.load_stimulus_table(nwb)
    spont = vn.spontaneous_block(nwb)
    running = vn.load_running_speed(nwb)
    t, blank = vn.stimulus_trials(stim, "drifting_gratings_windowed", vn.DG_PARAM_COLUMNS)
    plane = vn.load_plane(nwb, vn.list_planes(nwb)[0], trace_types=("events",))
    window_variants = {}
    for secs in (2.0, 1.985):
        cfg = sm.MetricConfig(dg_response_seconds=secs)
        window_variants[secs] = sm.drifting_gratings_metrics(
            plane, t, blank, spont, running, dg_type="windowed", config=cfg,
            rng=np.random.default_rng(SEEDS[0])).metrics
finally:
    io.close()

cmp_win = sm.compare_to_published(
    sm.to_published_schema(window_variants[2.0], "drifting_gratings_windowed"),
    sm.to_published_schema(window_variants[1.985], "drifting_gratings_windowed"),
    DG_METRICS, exact=["preferred_dir", "preferred_sf"])
rows = [{"metric": m, "median_abs_diff": v.get("median_abs_diff"),
         "max_abs_diff": v.get("max_abs_diff"), "frac_exact": v.get("frac_exact"),
         "pearson_r": v.get("pearson_r")}
        for m, v in cmp_win["metrics"].items()]
print(f"2.0 s vs 1.985 s, one plane ({len(plane.roi)} ROIs):")
display(pd.DataFrame(rows))
del plane

2.0 s vs 1.985 s, one plane (409 ROIs):


,metric,median_abs_diff,max_abs_diff,frac_exact,pearson_r
0,dsi,0.001313,0.708326,NaN,0.967265
1,frac_responsive_trials,0.000000,0.250000,NaN,0.992128
2,gosi,0.002708,0.128137,NaN,0.996815
3,is_responsive,0.000000,1.000000,NaN,0.979679
4,lifetime_sparseness,0.000824,0.012752,NaN,0.999336
5,osi,0.004593,0.459783,NaN,0.987812
6,preferred_dir,0.000000,270.000000,0.973105,0.971493
7,preferred_sf,0.000000,0.040000,0.987775,0.971278
8,pref_dir_mean,0.645873,355.031489,NaN,0.952081


In [8]:
for dg_type in ("full", "windowed"):
    fam = f"drifting_gratings_{dg_type}"
    out = sm.to_published_schema(dg_new[SEEDS[0]][dg_type], fam)
    out.to_csv(pjoin(save_dir, f"{fam}_M409828.csv"), index=False)
    print(f"wrote {fam}_M409828.csv  ({len(out)} rows)")

ssi_out = sm.to_published_schema(ssi_new[SEEDS[0]], "surround_supression_index")
ssi_out.to_csv(pjoin(save_dir, "surround_supression_index_M409828.csv"), index=False)
print(f"wrote surround_supression_index_M409828.csv  ({len(ssi_out)} rows)")

checkpoint("dg_validation",
           {"seeds": list(SEEDS), "window_seconds": 2.0,
            "window_comparison_2p0_vs_1p985": cmp_win, **dg_reports},
           save_dir, seed=SEEDS[0],
           sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])
if ssi_report is not None:
    checkpoint("ssi_validation", {"seeds": list(SEEDS), **ssi_report}, save_dir,
               seed=SEEDS[0],
               sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()])

# per-ROI differences for drilling in
if published_dir is not None:
    for fam, new in (("drifting_gratings_windowed", dg_new[SEEDS[0]]["windowed"]),
                     ("surround_supression_index", ssi_new[SEEDS[0]])):
        cols = DG_METRICS if fam.startswith("drifting") else sm.SSI_COLUMNS
        merged = sm.to_published_schema(new, fam).merge(
            sm.load_published(published_dir, fam),
            on=["column", "volume", "plane", "roi"], how="inner", suffixes=("_new", "_pub"))
        keep = ["column", "volume", "plane", "roi"]
        for m in cols:
            merged[f"{m}_diff"] = (pd.to_numeric(merged[f"{m}_new"], errors="coerce")
                                   - pd.to_numeric(merged[f"{m}_pub"], errors="coerce"))
            keep += [f"{m}_new", f"{m}_pub", f"{m}_diff"]
        path = pjoin(save_dir, "checks", f"{fam}_per_roi.csv")
        merged[keep].to_csv(path, index=False)
        print(f"wrote {path}  ({len(merged)} rows)")

wrote drifting_gratings_full_M409828.csv  (3673 rows)
wrote drifting_gratings_windowed_M409828.csv  (3673 rows)
wrote surround_supression_index_M409828.csv  (3673 rows)
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/dg_validation.json  (17.2 KB)
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/ssi_validation.json  (5.7 KB)
wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/drifting_gratings_windowed_per_roi.csv  (3673 rows)
wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/surround_supression_index_per_roi.csv  (3673 rows)


## Next

**M5 — natural images and natural images 12**, which reuse the drifting-gratings machinery
with `image_index` as the condition. Then **M6 — receptive fields**, where M1 already
settled the two open questions: the template is natively 8x14 (no downsampling needed) and
its pixel values are -1 / 0 / 1 rather than the 0 / 127 / 255 the original hard-codes.